In [ ]:
# ============================================================
# METRICS FOR THE w = 20 BACKBONE
# New CRS with local semantic filtering
# ============================================================

import networkx as nx
import pandas as pd
import numpy as np
import random

# ------------------------------------------------------------
# 0) CONFIGURATION
# ------------------------------------------------------------

W_SELECTED = 20
OUT_DIR = "YOUR_OUTPUT"

assert "G_rizoma" in globals(), "Primero debes ejecutar la construcción de G_rizoma."

# ------------------------------------------------------------
# 1) BUILD BACKBONE w = 20
# ------------------------------------------------------------

def build_backbone(G, wmin):
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from([
        (u, v, d)
        for u, v, d in G.edges(data=True)
        if float(d.get("weight", 1)) >= wmin
    ])
    H.remove_nodes_from([n for n in list(H.nodes()) if H.degree(n) == 0])
    H.remove_edges_from(nx.selfloop_edges(H))
    return H

H = build_backbone(G_rizoma, W_SELECTED)

components = list(nx.connected_components(H))
lcc_nodes = max(components, key=len)
L = H.subgraph(lcc_nodes).copy()

print("="*70)
print(f"BACKBONE w >= {W_SELECTED}")
print("="*70)
print("Nodes:", H.number_of_nodes())
print("Edges:", H.number_of_edges())
print("Components:", len(components))
print("LCC nodes:", L.number_of_nodes())
print("LCC edges:", L.number_of_edges())
print("LCC proportion:", L.number_of_nodes() / H.number_of_nodes())
print("Density:", nx.density(H))
print("Density LCC:", nx.density(L))

# ------------------------------------------------------------
# 2) DEGREE AND WEIGHTED DEGREE
# ------------------------------------------------------------

deg = dict(H.degree())
wdeg = dict(H.degree(weight="weight"))

df_nodes = pd.DataFrame({
    "node": list(H.nodes()),
    "degree": [deg[n] for n in H.nodes()],
    "weighted_degree": [wdeg[n] for n in H.nodes()]
})

avg_degree = df_nodes["degree"].mean()
avg_wdegree = df_nodes["weighted_degree"].mean()

def top_concentration(df, col, pct):
    df_sorted = df.sort_values(col, ascending=False)
    k = max(1, int(np.ceil(len(df_sorted) * pct)))
    return df_sorted.head(k)[col].sum() / df_sorted[col].sum()

conc_1 = top_concentration(df_nodes, "degree", 0.01)
conc_5 = top_concentration(df_nodes, "degree", 0.05)
conc_10 = top_concentration(df_nodes, "degree", 0.10)

print("\n--- Degree summary ---")
print("Average degree:", avg_degree)
print("Average weighted degree:", avg_wdegree)
print("Top 1% degree concentration:", conc_1)
print("Top 5% degree concentration:", conc_5)
print("Top 10% degree concentration:", conc_10)

top_degree = df_nodes.sort_values("degree", ascending=False).head(20)
top_wdegree = df_nodes.sort_values("weighted_degree", ascending=False).head(20)

print("\nTop degree:")
print(top_degree.to_string(index=False))

print("\nTop weighted degree:")
print(top_wdegree.to_string(index=False))

# ------------------------------------------------------------
# 3) DISTANCES IN THE LCC
# ------------------------------------------------------------

avg_shortest_path = nx.average_shortest_path_length(L)
diameter = nx.diameter(L)
radius = nx.radius(L)

print("\n--- Distance metrics on LCC ---")
print("Average shortest path:", avg_shortest_path)
print("Diameter:", diameter)
print("Radius:", radius)

# Example of a diametral path
ecc = nx.eccentricity(L)
u = max(ecc, key=ecc.get)
lengths = nx.single_source_shortest_path_length(L, u)
v = max(lengths, key=lengths.get)
diam_path = nx.shortest_path(L, u, v)

print("Example diametral path:")
print(" -> ".join(diam_path))

# ------------------------------------------------------------
# 4) LOUVAIN COMMUNITIES
# ------------------------------------------------------------

try:
    import community as community_louvain

    part = community_louvain.best_partition(H, weight="weight", random_state=42)
    nx.set_node_attributes(H, part, "community")

    communities_dict = {}
    for node, c in part.items():
        communities_dict.setdefault(c, []).append(node)

    communities = [set(nodes) for nodes in communities_dict.values()]
    modularity = nx.community.modularity(H, communities, weight="weight")
    community_sizes = sorted([len(c) for c in communities], reverse=True)

    print("\n--- Louvain communities ---")
    print("Number of communities:", len(communities))
    print("Modularity:", modularity)
    print("Largest community sizes:", community_sizes[:10])

except Exception as e:
    print("Louvain failed:", e)
    part = {n: 0 for n in H.nodes()}
    communities = [set(H.nodes())]
    modularity = None
    community_sizes = [H.number_of_nodes()]

# ------------------------------------------------------------
# 5) APPROXIMATE BETWEENNESS IN THE BACKBONE
# ------------------------------------------------------------

bet = nx.betweenness_centrality(H, normalized=True, weight=None)

df_nodes["betweenness"] = [bet.get(n, np.nan) for n in df_nodes["node"]]

top_betweenness = df_nodes.sort_values("betweenness", ascending=False).head(20)

print("\nTop betweenness:")
print(top_betweenness.to_string(index=False))

# ------------------------------------------------------------
# 6) EIGENVECTOR IN THE LCC
# ------------------------------------------------------------

eig = nx.eigenvector_centrality(L, max_iter=2000, weight="weight")

df_nodes["eigenvector_lcc"] = [
    eig.get(n, np.nan) for n in df_nodes["node"]
]

top_eigenvector = df_nodes.sort_values("eigenvector_lcc", ascending=False).head(20)

print("\nTop eigenvector:")
print(top_eigenvector.to_string(index=False))

# ------------------------------------------------------------
# 7) K-CORE
# ------------------------------------------------------------

core = nx.core_number(H)
df_nodes["core_number"] = [core.get(n, np.nan) for n in df_nodes["node"]]

max_k = max(core.values())
G_core = nx.k_core(H, k=max_k)

print("\n--- K-core ---")
print("Max k-core:", max_k)
print("Max core nodes:", G_core.number_of_nodes())
print("Max core edges:", G_core.number_of_edges())

top_kcore = df_nodes.sort_values(
    ["core_number", "degree"],
    ascending=[False, False]
).head(30)

print("\nTop k-core:")
print(top_kcore[["node", "core_number", "degree", "weighted_degree"]].to_string(index=False))

# ------------------------------------------------------------
# 8) CONNECTIVITY, BRIDGES, AND ARTICULATION POINTS
# ------------------------------------------------------------

node_conn = nx.node_connectivity(L)
edge_conn = nx.edge_connectivity(L)
bridges = list(nx.bridges(L))
art_points = list(nx.articulation_points(L))

print("\n--- Connectivity ---")
print("Node connectivity LCC:", node_conn)
print("Edge connectivity LCC:", edge_conn)
print("Number of bridges LCC:", len(bridges))
print("Number of articulation points LCC:", len(art_points))

# ------------------------------------------------------------
# 9) SUMMARY TABLE FOR THE PAPER
# ------------------------------------------------------------

summary = {
    "w": W_SELECTED,
    "nodes": H.number_of_nodes(),
    "edges": H.number_of_edges(),
    "components": len(components),
    "lcc_nodes": L.number_of_nodes(),
    "lcc_edges": L.number_of_edges(),
    "lcc_prop": L.number_of_nodes() / H.number_of_nodes(),
    "density": nx.density(H),
    "density_lcc": nx.density(L),
    "avg_degree": avg_degree,
    "avg_weighted_degree": avg_wdegree,
    "top_1_degree_concentration": conc_1,
    "top_5_degree_concentration": conc_5,
    "top_10_degree_concentration": conc_10,
    "avg_shortest_path_lcc": avg_shortest_path,
    "diameter_lcc": diameter,
    "radius_lcc": radius,
    "n_communities": len(communities),
    "modularity": modularity,
    "largest_community_1": community_sizes[0] if len(community_sizes) > 0 else None,
    "largest_community_2": community_sizes[1] if len(community_sizes) > 1 else None,
    "largest_community_3": community_sizes[2] if len(community_sizes) > 2 else None,
    "max_k_core": max_k,
    "max_core_nodes": G_core.number_of_nodes(),
    "max_core_edges": G_core.number_of_edges(),
    "node_connectivity_lcc": node_conn,
    "edge_connectivity_lcc": edge_conn,
    "n_bridges_lcc": len(bridges),
    "n_articulation_points_lcc": len(art_points),
    "diametral_path_example": " -> ".join(diam_path)
}

df_summary_w20 = pd.DataFrame([summary])

print("\n" + "="*70)
print("SUMMARY FOR PAPER")
print("="*70)
print(df_summary_w20.T.to_string())

# ------------------------------------------------------------
# 10) EXPORT
# ------------------------------------------------------------

df_summary_w20.to_csv(f"{OUT_DIR}/summary_backbone_w20_new_crs.csv", index=False)
df_nodes.to_csv(f"{OUT_DIR}/node_metrics_backbone_w20_new_crs.csv", index=False)
top_degree.to_csv(f"{OUT_DIR}/top_degree_w20_new_crs.csv", index=False)
top_wdegree.to_csv(f"{OUT_DIR}/top_weighted_degree_w20_new_crs.csv", index=False)
top_betweenness.to_csv(f"{OUT_DIR}/top_betweenness_w20_new_crs.csv", index=False)
top_eigenvector.to_csv(f"{OUT_DIR}/top_eigenvector_w20_new_crs.csv", index=False)
top_kcore.to_csv(f"{OUT_DIR}/top_kcore_w20_new_crs.csv", index=False)

print("\nArchivos exportados al escritorio.")